In [1]:
import json
import time
from collections import deque
from datetime import datetime, timedelta, timezone

import pandas as pd
from kafka import KafkaConsumer
from IPython.display import display, HTML, clear_output

# -------- CONFIG --------
BOOTSTRAP = "localhost:9092"
TOPIC = "sensor_topic"
MODEL_PATH = "/home/osbdet/notebooks/Group Project IOT/anomaly_model_config.json"

WINDOW_SECONDS = 10
REFRESH_SECONDS = 1.0

def utc_now():
    return datetime.now(timezone.utc)

def ms_to_utc_datetime(ms: int) -> datetime:
    return datetime.fromtimestamp(ms / 1000.0, tz=timezone.utc)

# -------- Load model config --------
with open(MODEL_PATH, "r") as f:
    cfg = json.load(f)

W = int(cfg["window_size"])
k = float(cfg["k_multiplier"])
temp_resid_std = float(cfg["temp_resid_std"])
vib_resid_std = float(cfg["vib_resid_std"])

TEMP_THRESH = k * temp_resid_std
VIB_THRESH  = k * vib_resid_std

# -------- Online rolling residual scorer (per device) --------
temp_buf = {}
vib_buf = {}

def get_bufs(device_id: str):
    if device_id not in temp_buf:
        temp_buf[device_id] = deque(maxlen=W)
        vib_buf[device_id] = deque(maxlen=W)
    return temp_buf[device_id], vib_buf[device_id]

def score(device_id: str, temp_c: float, vib: float):
    tbuf, vbuf = get_bufs(device_id)
    warmup_min = max(5, W // 3)

    if len(tbuf) < warmup_min or len(vbuf) < warmup_min:
        tbuf.append(temp_c); vbuf.append(vib)
        return "Normal", 0.0, 0.0

    tmean = sum(tbuf) / len(tbuf)
    vmean = sum(vbuf) / len(vbuf)

    temp_resid = temp_c - tmean
    vib_resid  = vib - vmean

    status = "Anomaly" if (abs(temp_resid) > TEMP_THRESH or abs(vib_resid) > VIB_THRESH) else "Normal"

    tbuf.append(temp_c); vbuf.append(vib)
    return status, float(temp_resid), float(vib_resid)

# -------- Kafka consumer --------
consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=BOOTSTRAP,
    value_deserializer=lambda m: json.loads(m.decode("utf-8")),
    auto_offset_reset="latest",
    enable_auto_commit=False,
    group_id=f"notebook-ui-{int(time.time())}"
)

events = []
print("▶️ Dashboard running. Interrupt cell to stop.")

# CSS for a "website" look
DASH_CSS = """
<style>
  .dash-wrap { font-family: ui-sans-serif, system-ui, -apple-system, Segoe UI, Roboto, Arial;
               padding: 14px; max-width: 1100px; margin: 0 auto; }
  .dash-header { display:flex; justify-content:space-between; align-items:center;
                 background: #0f172a; color: white; padding: 12px 14px; border-radius: 12px; }
  .dash-title { font-size: 18px; font-weight: 700; letter-spacing: 0.2px; }
  .dash-sub { font-size: 12px; opacity: 0.85; }
  .cards { display:grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin-top: 10px; }
  .card { background: white; border: 1px solid #e5e7eb; border-radius: 12px; padding: 10px 12px;
          box-shadow: 0 1px 2px rgba(0,0,0,0.05); }
  .card .label { font-size: 12px; color: #64748b; }
  .card .value { font-size: 18px; font-weight: 700; margin-top: 4px; color: #0f172a; }
  .card .hint { font-size: 11px; color: #64748b; margin-top: 4px; }
  .pill { display:inline-block; padding: 2px 8px; border-radius: 999px; font-size: 12px;
          border: 1px solid rgba(255,255,255,0.2); background: rgba(255,255,255,0.08); }
  .table-wrap { margin-top: 10px; background: white; border: 1px solid #e5e7eb; border-radius: 12px;
                overflow:hidden; box-shadow: 0 1px 2px rgba(0,0,0,0.05); }
  table { width: 100%; border-collapse: collapse; }
  thead { background: #f8fafc; }
  th, td { padding: 10px 10px; border-bottom: 1px solid #e5e7eb; font-size: 13px; text-align:left; }
  th { font-size: 12px; text-transform: uppercase; letter-spacing: .04em; color: #475569; }
  tr:hover td { background: #f8fafc; }
  .ok { color: #166534; font-weight: 700; }
  .bad { color: #991b1b; font-weight: 700; }
  .muted { color: #64748b; }
  .footer { margin-top: 10px; font-size: 12px; color: #64748b; }
</style>
"""

def render_dashboard(df_show: pd.DataFrame, now: datetime, polled_count: int, anoms: int):
    # Convert datetimes to strings for neat display
    d = df_show.copy()
    for col in ["event_time", "ingest_time"]:
        d[col] = d[col].astype(str)

    # Build rows HTML
    rows_html = ""
    for _, r in d.iterrows():
        status_cls = "bad" if "Anomaly" in str(r["status"]) else "ok"
        rows_html += f"""
          <tr>
            <td>{r['device_id']}</td>
            <td class="muted">{r['event_time']}</td>
            <td class="muted">{r['ingest_time']}</td>
            <td>{r['temp_c']}</td>
            <td>{r['vib']}</td>
            <td class="{status_cls}">{r['status']}</td>
          </tr>
        """

    html = f"""
    {DASH_CSS}
    <div class="dash-wrap">
      <div class="dash-header">
        <div>
          <div class="dash-title">IoT Machine Monitor</div>
          <div class="dash-sub">Kafka → Notebook Dashboard (last {WINDOW_SECONDS}s, by ingest time)</div>
        </div>
        <div class="pill">Topic: <b>{TOPIC}</b> &nbsp;|&nbsp; Bootstrap: <b>{BOOTSTRAP}</b></div>
      </div>

      <div class="cards">
        <div class="card">
          <div class="label">Now (UTC)</div>
          <div class="value">{now.strftime('%Y-%m-%d %H:%M:%S')}</div>
          <div class="hint">Refresh every {REFRESH_SECONDS:.1f}s</div>
        </div>
        <div class="card">
          <div class="label">Polled this tick</div>
          <div class="value">{polled_count}</div>
          <div class="hint">Messages fetched this refresh</div>
        </div>
        <div class="card">
          <div class="label">Rows in window</div>
          <div class="value">{len(df_show)}</div>
          <div class="hint">Last {WINDOW_SECONDS}s ingested</div>
        </div>
        <div class="card">
          <div class="label">Anomalies in window</div>
          <div class="value">{anoms}</div>
          <div class="hint">Thresholds: temp={TEMP_THRESH:.3f}, vib={VIB_THRESH:.3f}</div>
        </div>
      </div>

      <div class="table-wrap">
        <table>
          <thead>
            <tr>
              <th>Device</th>
              <th>Event Time</th>
              <th>Ingest Time</th>
              <th>Temp (°C)</th>
              <th>Vibration</th>
              <th>Status</th>
            </tr>
          </thead>
          <tbody>
            {rows_html if rows_html else '<tr><td colspan="6" class="muted">Waiting for new Kafka messages…</td></tr>'}
          </tbody>
        </table>
      </div>

      <div class="footer">
        Anomaly logic: residual vs rolling mean (window={W}), anomaly if |residual| &gt; k·std (k={k}).
      </div>
    </div>
    """
    return HTML(html)

while True:
    msg_pack = consumer.poll(timeout_ms=1000, max_records=500)

    polled_count = 0
    for tp, records in msg_pack.items():
        polled_count += len(records)
        for record in records:
            p = record.value

            device_id = str(p.get("device_id", "unknown"))
            ingest_time = utc_now()

            try:
                event_time = ms_to_utc_datetime(int(p["event_time_ms"]))
            except Exception:
                event_time = None

            temp_c = float(p.get("temp_c", 0.0))
            vib = float(p.get("vib", 0.0))

            status, temp_resid, vib_resid = score(device_id, temp_c, vib)

            events.append({
                "device_id": device_id,
                "event_time": event_time,
                "ingest_time": ingest_time,
                "temp_c": temp_c,
                "vib": vib,
                "status": ("🔴 Anomaly" if status == "Anomaly" else "✅ Normal"),
                "temp_resid": temp_resid,
                "vib_resid": vib_resid
            })

    now = utc_now()
    cutoff = now - timedelta(seconds=WINDOW_SECONDS)
    recent = [e for e in events if e["ingest_time"] >= cutoff]
    df = pd.DataFrame(recent)

    clear_output(wait=True)

    if df.empty:
        display(render_dashboard(pd.DataFrame(columns=["device_id","event_time","ingest_time","temp_c","vib","status"]),
                                 now, polled_count, 0))
        time.sleep(REFRESH_SECONDS)
        continue

    anoms = int(df["status"].str.contains("Anomaly").sum())
    df_show = df.sort_values("ingest_time", ascending=False)[
        ["device_id", "event_time", "ingest_time", "temp_c", "vib", "status"]
    ].head(20)

    display(render_dashboard(df_show, now, polled_count, anoms))
    time.sleep(REFRESH_SECONDS)

KeyboardInterrupt: 